In [ ]:
# GCN Regression Model for Calixarene Binding Prediction
# This notebook implements a Graph Convolutional Network for predicting calixarene binding affinities

# Standard library imports
import os
import sys
import time
import pickle
import copy
from collections import defaultdict

# Third-party imports
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from sklearn.model_selection import train_test_split, LeaveOneOut
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
import seaborn as sns
from rdkit import Chem
from rdkit.Chem import AllChem, QED, rdMolDescriptors, MolSurf
from rdkit.Chem.Draw import SimilarityMaps, rdMolDraw2D
from rdkit.Chem import rdDepictor

# Custom imports
from GCN import Fingerprint, GCNModel, save_smiles_dicts, get_smiles_dicts, get_smiles_array

# Configuration
torch.backends.cudnn.benchmark = True
sys.setrecursionlimit(50000)
sns.set_style("darkgrid")

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

/usr/local/lib/python3.11/dist-packages/torch/__init__.py:614: UserWarning: torch.set_default_tensor_type() is deprecated as of PyTorch 2.1, please use torch.set_default_dtype() and torch.set_default_device() as alternatives. (Triggered internally at ../torch/csrc/tensor/python_tensor.cpp:451.)
  _C._set_default_tensor_type(t)


cuda


In [ ]:
def prepare_model_and_data(raw_filename, target_name='Calx', targets=None, batch_size=50, 
                          epochs=800, p_dropout=0.1, fingerprint_dim=150, weight_decay=2, 
                          learning_rate=3, radius=3, T=2, per_target_output_units_num=1):
    """
    Prepare the GCN model and process input data for training.
    
    Parameters:
    -----------
    raw_filename : str
        Path to the CSV file containing SMILES and target data
    target_name : str, default='Calx'
        Name of the target column
    targets : list, optional
        List of target column names. If None, uses default histone modifications
    batch_size : int, default=50
        Batch size for training
    epochs : int, default=800
        Number of training epochs
    p_dropout : float, default=0.1
        Dropout probability
    fingerprint_dim : int, default=150
        Dimension of molecular fingerprint
    weight_decay : int, default=2
        Weight decay parameter (10^-weight_decay)
    learning_rate : int, default=3
        Learning rate parameter (10^-learning_rate)
    radius : int, default=3
        Radius for molecular graph convolution
    T : int, default=2
        Number of graph convolution layers
    per_target_output_units_num : int, default=1
        Number of output units per target
    
    Returns:
    --------
    tuple
        (model, optimizer, loss_function, remained_df, targets, feature_dicts)
    """
    if targets is None:
        targets = ['H3K4', 'H3K4ac', 'H3K4me1', 'H3K4me2', 'H3K4me3', 'H3K9me3', 'H3R2me2a', 'H3R2me2s']

    feature_filename = raw_filename.replace('.csv', '.pickle')
    filename = raw_filename.replace('.csv', '')
    smiles_targets_df = pd.read_csv(raw_filename)

    smilesList = smiles_targets_df.SMILES.values
    print(f"Number of input SMILES: {len(smilesList)}")

    # Process SMILES and convert to canonical form
    remained_smiles = []
    canonical_smiles_list = []
    for smiles in smilesList:
        try:
            mol = Chem.MolFromSmiles(smiles)
            if mol is not None:
                remained_smiles.append(smiles)
                canonical_smiles_list.append(Chem.MolToSmiles(mol, isomericSmiles=True))
        except Exception as e:
            print(f"Failed to process SMILES: {smiles}")
            continue
    
    print(f"Successfully processed SMILES: {len(remained_smiles)}")

    # Filter dataframe and add canonical SMILES
    smiles_targets_df = smiles_targets_df[smiles_targets_df["SMILES"].isin(remained_smiles)]
    smiles_targets_df['cano_smiles'] = canonical_smiles_list

    output_units_num = len(targets) * per_target_output_units_num

    # Load or create molecular features
    if os.path.isfile(feature_filename):
        feature_dicts = pickle.load(open(feature_filename, "rb"))
    else:
        feature_dicts = save_smiles_dicts(smilesList, filename)
    feature_dicts = get_smiles_dicts(smilesList)

    # Filter data based on available features
    remained_df = smiles_targets_df[smiles_targets_df["cano_smiles"].isin(feature_dicts['smiles_to_atom_mask'].keys())]

    # Get feature dimensions
    x_atom, x_bonds, x_atom_index, x_bond_index, x_mask, _ = get_smiles_array([canonical_smiles_list[0]], feature_dicts)
    num_atom_features = x_atom.shape[-1]
    num_bond_features = x_bonds.shape[-1]

    # Initialize model and training components
    loss_function = nn.MSELoss()
    model = GCNModel(radius, T, num_atom_features, num_bond_features, fingerprint_dim, output_units_num, p_dropout)
    model.to(device)

    optimizer = optim.Adam(model.parameters(), 10**-learning_rate, weight_decay=10**-weight_decay)

    return model, optimizer, loss_function, remained_df, targets, feature_dicts



In [ ]:
def train(model, dataset, optimizer, loss_function, epoch, batch_size, targets, feature_dicts, ratio_list):
    """
    Train the GCN model for one epoch.
    
    Parameters:
    -----------
    model : GCNModel
        The GCN model to train
    dataset : pd.DataFrame
        Training dataset
    optimizer : torch.optim.Optimizer
        Optimizer for training
    loss_function : torch.nn.Module
        Loss function
    epoch : int
        Current epoch number
    batch_size : int
        Batch size for training
    targets : list
        List of target column names
    feature_dicts : dict
        Dictionary containing molecular features
    ratio_list : list
        List of ratios for weighting losses
    
    Returns:
    --------
    float
        Average training loss for the epoch
    """
    model.train()
    
    # Create batches
    if len(dataset) <= batch_size:
        batch_list = [dataset.index]
    else:
        valList = np.arange(0, dataset.shape[0])
        np.random.shuffle(valList)
        batch_list = [valList[i:i+batch_size] for i in range(0, dataset.shape[0], batch_size)]
    
    device = next(model.parameters()).device
    total_loss = 0
    
    for batch in batch_list:
        batch_df = dataset.loc[batch, :]
        smiles_list = batch_df.cano_smiles.values

        # Convert molecules to tensors
        x_atom, x_bonds, x_atom_index, x_bond_index, x_mask, _ = get_smiles_array(smiles_list, feature_dicts)
        
        x_atom = torch.Tensor(x_atom).to(device)
        x_bonds = torch.Tensor(x_bonds).to(device)
        x_atom_index = torch.LongTensor(x_atom_index).to(device)
        x_bond_index = torch.LongTensor(x_bond_index).to(device)
        x_mask = torch.Tensor(x_mask).to(device)
        
        # Forward pass
        atoms_prediction, mol_prediction = model(x_atom, x_bonds, x_atom_index, x_bond_index, x_mask)
        
        # Calculate loss
        optimizer.zero_grad()
        loss = 0.0
        
        for i, target in enumerate(targets):
            y_pred = mol_prediction[:, i]
            y_val = torch.Tensor(batch_df[target].values).squeeze().to(device)
            target_loss = loss_function(y_pred, y_val) * (ratio_list[i] ** 2)
            loss += target_loss
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(batch_list)

def eval(model, dataset, targets, feature_dicts, batch_size):
    """
    Evaluate the GCN model on a dataset.
    
    Parameters:
    -----------
    model : GCNModel
        The GCN model to evaluate
    dataset : pd.DataFrame
        Dataset to evaluate on
    targets : list
        List of target column names
    feature_dicts : dict
        Dictionary containing molecular features
    batch_size : int
        Batch size for evaluation
    
    Returns:
    --------
    tuple
        (eval_MAE, eval_MSE, y_pred_list, y_val_list, smiles_list)
    """
    model.eval()
    
    # Initialize containers for metrics and predictions
    eval_MAE_list = {target: [] for target in targets}
    eval_MSE_list = {target: [] for target in targets}
    y_val_list = {target: [] for target in targets}
    y_pred_list = {target: [] for target in targets}
    smiles_list = []
    
    # Create batches
    if len(dataset) <= batch_size:
        batch_list = [dataset.index]
    else:
        valList = np.arange(0, dataset.shape[0])
        batch_list = [valList[i:i+batch_size] for i in range(0, dataset.shape[0], batch_size)]
    
    device = next(model.parameters()).device
    
    with torch.no_grad():
        for batch in batch_list:
            batch_df = dataset.loc[batch, :]
            batch_smiles = batch_df.cano_smiles.values
            smiles_list.extend(batch_smiles)
            
            # Convert molecules to tensors
            x_atom, x_bonds, x_atom_index, x_bond_index, x_mask, _ = get_smiles_array(batch_smiles, feature_dicts)
            
            x_atom = torch.Tensor(x_atom).to(device)
            x_bonds = torch.Tensor(x_bonds).to(device)
            x_atom_index = torch.LongTensor(x_atom_index).to(device)
            x_bond_index = torch.LongTensor(x_bond_index).to(device)
            x_mask = torch.Tensor(x_mask).to(device)
            
            # Forward pass
            atoms_prediction, mol_prediction = model(x_atom, x_bonds, x_atom_index, x_bond_index, x_mask)
            
            # Calculate metrics for each target
            for i, target in enumerate(targets):
                y_pred = mol_prediction[:, i].view(-1, 1)
                y_val = torch.Tensor(batch_df[target].values.reshape(-1, 1)).to(device)
                
                MAE = F.l1_loss(y_pred, y_val, reduction='none')
                MSE = F.mse_loss(y_pred, y_val, reduction='none')
                
                y_pred_list[target].extend(y_pred.cpu().numpy().flatten())
                y_val_list[target].extend(y_val.cpu().numpy().flatten())
                
                eval_MAE_list[target].extend(MAE.cpu().numpy().flatten())
                eval_MSE_list[target].extend(MSE.cpu().numpy().flatten())
    
    # Calculate average metrics across targets
    eval_MAE = np.array([np.mean(eval_MAE_list[target]) for target in targets])
    eval_MSE = np.array([np.mean(eval_MSE_list[target]) for target in targets])
    
    return eval_MAE, eval_MSE, y_pred_list, y_val_list, smiles_list

def train_and_evaluate(model, remained_df, targets, feature_dicts, optimizer, loss_function, 
                      batch_size, num_epochs, patience=30, min_delta=0.001, 
                      prefix_filename='', start_time=''):
    """
    Train and evaluate the GCN model using Leave-One-Out cross-validation.
    
    Parameters:
    -----------
    model : GCNModel
        The GCN model to train and evaluate
    remained_df : pd.DataFrame
        Dataset containing molecular features and targets
    targets : list
        List of target column names
    feature_dicts : dict
        Dictionary containing molecular features
    optimizer : torch.optim.Optimizer
        Optimizer for training
    loss_function : torch.nn.Module
        Loss function
    batch_size : int
        Batch size for training
    num_epochs : int
        Maximum number of training epochs
    patience : int, default=30
        Early stopping patience
    min_delta : float, default=0.001
        Minimum improvement for early stopping
    prefix_filename : str, default=''
        Prefix for output files
    start_time : str, default=''
        Start time string for logging
    
    Returns:
    --------
    dict
        Dictionary containing host predictions, overall metrics, fold results, 
        and train/test predictions
    """
    # Initialize tracking structures
    fold_results = []
    best_param = {
        "train_epoch": 0, 
        "val_epoch": 0, 
        "train_MSE": float('inf'), 
        "val_MSE": float('inf')
    }
    
    # Prediction containers
    prediction_containers = {
        'train': {t: {'pred': [], 'actual': []} for t in targets},
        'test': {t: {'pred': [], 'actual': []} for t in targets}
    }
    
    # Cross-validation setup
    loo = LeaveOneOut()
    initial_state = model.state_dict()
    host_results = {}
    
    for fold, (train_index, test_index) in enumerate(loo.split(remained_df)):
        # --- Fold Initialization ---
        # Reset model and optimizer
        model.load_state_dict(initial_state)
        optimizer = type(optimizer)(model.parameters(), **optimizer.defaults)  # Proper reset
        
        # Data splits
        train_df = remained_df.iloc[train_index]
        test_df = remained_df.iloc[test_index]
        current_smile = test_df['Host'].values[0]
        
        # Validation split with minimum size
        val_size = max(5, int(0.1 * len(train_df)))
        train_df, val_df = train_test_split(
            train_df, 
            test_size=val_size,
            stratify=pd.qcut(train_df[targets[0]], q=5) if len(targets) > 0 else None
        )
        
        # --- Training Loop ---
        best_val_mse = float('inf')
        epochs_no_improve = 0
        best_model_state = None
        best_val_metrics = None
        
        for epoch in range(num_epochs):
            # Training phase
            train_loss = train(model, train_df, optimizer, loss_function, epoch, 
                              batch_size, targets, feature_dicts, ratio_list=[1.0]*len(targets))
            
            # Validation evaluation
            val_metrics = eval(model, val_df, targets, feature_dicts, batch_size)
            val_mae, val_mse = val_metrics[0].mean(), val_metrics[1].mean()
            
            # Model checkpointing based on validation
            if val_mse < best_val_mse - min_delta:
                best_val_mse = val_mse
                epochs_no_improve = 0
                best_model_state = model.state_dict()
                best_val_metrics = val_metrics
                best_param.update({
                    "val_epoch": epoch,
                    "val_MSE": val_mse,
                    "train_epoch": epoch,
                    "train_MSE": train_loss  # Using actual training loss
                })
            else:
                epochs_no_improve += 1
            
            if epochs_no_improve >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break
        
        # --- Post-Training Evaluation ---
        # Load best model
        model.load_state_dict(best_model_state)
        
        # Re-evaluate all sets with best model
        final_train_metrics = eval(model, train_df, targets, feature_dicts, batch_size)
        final_val_metrics = best_val_metrics
        test_metrics = eval(model, test_df, targets, feature_dicts, batch_size)
        test_MAE, test_MSE, test_pred, test_actual, _ = test_metrics
        
        # Store results by host
        host_results[current_smile] = {
            target: {
                "actual": float(test_actual[target][0]),  # Convert to Python float
                "predicted": float(test_pred[target][0])
            }
            for target in targets
        }
        # Store predictions
        for target in targets:
            # Training data (from final evaluation)
            prediction_containers['train'][target]['pred'].extend(final_train_metrics[2][target])
            prediction_containers['train'][target]['actual'].extend(final_train_metrics[3][target])
            
            # Test data
            prediction_containers['test'][target]['pred'].extend(test_metrics[2][target])
            prediction_containers['test'][target]['actual'].extend(test_metrics[3][target])
        
        # --- Fold Bookkeeping ---
        fold_results.append({
            'fold': fold + 1,
            'smiles': current_smile,
            'train_metrics': {
                'MAE': final_train_metrics[0].tolist(),
                'MSE': final_train_metrics[1].tolist()
            },
            'val_metrics': {
                'MAE': final_val_metrics[0].tolist(),
                'MSE': final_val_metrics[1].tolist()
            },
            'test_metrics': {
                'MAE': test_metrics[0].tolist(),
                'MSE': test_metrics[1].tolist()
            }
        })
        
        print(f"\nFold {fold+1} Summary:")
        print(f"Train MSE: {np.mean(final_train_metrics[1]):.4f}")
        print(f"Val MSE: {np.mean(final_val_metrics[1]):.4f}")
        print(f"Test MSE: {np.mean(test_metrics[1]):.4f}")
    
    # --- Final Aggregation ---
    def calculate_overall(metric_key):
        train_vals = [np.mean(f['train_metrics'][metric_key]) for f in fold_results]
        test_vals = [np.mean(f['test_metrics'][metric_key]) for f in fold_results]
        return np.mean(train_vals), np.mean(test_vals)
    
    overall_train_mae, overall_test_mae = calculate_overall('MAE')
    overall_train_mse, overall_test_mse = calculate_overall('MSE')
    
    print("\nFinal Performance:")
    print(f"Train MAE: {overall_train_mae:.4f} ± {np.std([f['train_metrics']['MAE'] for f in fold_results]):.4f}")
    print(f"Test MAE: {overall_test_mae:.4f} ± {np.std([f['test_metrics']['MAE'] for f in fold_results]):.4f}")
    print(f"Train MSE: {overall_train_mse:.4f} ± {np.std([f['train_metrics']['MSE'] for f in fold_results]):.4f}")
    print(f"Test MSE: {overall_test_mse:.4f} ± {np.std([f['test_metrics']['MSE'] for f in fold_results]):.4f}")
    
    return {
        "host_predictions": host_results,  
        "overall_metrics": (overall_train_mae, overall_train_mse, 
                           overall_test_mae, overall_test_mse),
        "fold_results": fold_results,
        "train_predictions": prediction_containers['train'],
        "test_predictions": prediction_containers['test']
    }


In [ ]:
# Configuration parameters
DATA_PATH = '../Database/calix smiles small set.csv'  # Update this path as needed
BATCH_SIZE = 38
NUM_EPOCHS = 800
DROPOUT = 0.1
FINGERPRINT_DIM = 150
WEIGHT_DECAY = 2
LEARNING_RATE = 3
RADIUS = 3
T = 2

# Prepare model and data
model, optimizer, loss_function, remained_df, targets, feature_dicts = prepare_model_and_data(
    raw_filename=DATA_PATH,
    target_name='Calx', 
    targets=None, 
    batch_size=BATCH_SIZE, 
    epochs=NUM_EPOCHS,
    p_dropout=DROPOUT, 
    fingerprint_dim=FINGERPRINT_DIM, 
    weight_decay=WEIGHT_DECAY, 
    learning_rate=LEARNING_RATE, 
    radius=RADIUS,
    T=T, 
    per_target_output_units_num=1
)

# Train and evaluate the model
results = train_and_evaluate(
    model=model, 
    remained_df=remained_df, 
    targets=targets, 
    feature_dicts=feature_dicts, 
    optimizer=optimizer, 
    loss_function=loss_function, 
    batch_size=BATCH_SIZE, 
    num_epochs=NUM_EPOCHS
)                                                                        

number of all smiles:  28
number of successfully processed smiles:  28
Early stopping at epoch 127

Fold 1 Summary:
Train MSE: 1.2378
Val MSE: 0.5953
Test MSE: 2.0249
Early stopping at epoch 47

Fold 2 Summary:
Train MSE: 0.6932
Val MSE: 1.4436
Test MSE: 4.8550
Early stopping at epoch 95

Fold 3 Summary:
Train MSE: 0.8085
Val MSE: 0.8881
Test MSE: 2.1530
Early stopping at epoch 56

Fold 4 Summary:
Train MSE: 0.7257
Val MSE: 1.1898
Test MSE: 1.4640
Early stopping at epoch 63

Fold 5 Summary:
Train MSE: 0.5913
Val MSE: 0.7017
Test MSE: 1.2367
Early stopping at epoch 84

Fold 6 Summary:
Train MSE: 0.4676
Val MSE: 0.8673
Test MSE: 0.4145
Early stopping at epoch 41

Fold 7 Summary:
Train MSE: 0.4345
Val MSE: 0.4616
Test MSE: 0.2981
Early stopping at epoch 71

Fold 8 Summary:
Train MSE: 0.2886
Val MSE: 1.1583
Test MSE: 1.5956
Early stopping at epoch 56

Fold 9 Summary:
Train MSE: 0.3322
Val MSE: 0.6992
Test MSE: 1.6063
Early stopping at epoch 35

Fold 10 Summary:
Train MSE: 0.4089
Val MSE: 0

In [ ]:
# Display results
print("=" * 80)
print("GCN REGRESSION RESULTS - HOST-SPECIFIC PREDICTIONS")
print("=" * 80)

host_preds = results["host_predictions"]

# Display predictions for each host
for smiles, preds in host_preds.items():
    print(f"\nHost: {smiles}")
    print("-" * 50)
    for target_name, values in preds.items():
        actual = values['actual']
        predicted = values['predicted']
        error = abs(actual - predicted)
        print(f"{target_name:>10}: Actual={actual:>8.4f}, Predicted={predicted:>8.4f}, Error={error:>7.4f}")

# Display overall performance metrics
print("\n" + "=" * 80)
print("OVERALL PERFORMANCE METRICS")
print("=" * 80)

overall_metrics = results["overall_metrics"]
train_mae, train_mse, test_mae, test_mse = overall_metrics

print(f"Training Performance:")
print(f"  Mean Absolute Error (MAE): {train_mae:.4f}")
print(f"  Mean Squared Error (MSE):  {train_mse:.4f}")
print(f"\nTest Performance:")
print(f"  Mean Absolute Error (MAE): {test_mae:.4f}")
print(f"  Mean Squared Error (MSE):  {test_mse:.4f}")
print(f"  Root Mean Squared Error:   {np.sqrt(test_mse):.4f}")


Host: AP8
H3K4:
  Actual: -1.3093
  Predicted: -0.2797
  Error: 1.0296
H3K4ac:
  Actual: 0.0000
  Predicted: -0.1835
  Error: 0.1835
H3K4me1:
  Actual: -3.0791
  Predicted: -1.4237
  Error: 1.6554
H3K4me2:
  Actual: -5.1850
  Predicted: -3.3079
  Error: 1.8771
H3K4me3:
  Actual: -5.0207
  Predicted: -3.2439
  Error: 1.7768
H3K9me3:
  Actual: -3.7723
  Predicted: -2.8747
  Error: 0.8975
H3R2me2a:
  Actual: -4.1997
  Predicted: -2.5113
  Error: 1.6884
H3R2me2s:
  Actual: -2.6037
  Predicted: -1.1795
  Error: 1.4242

Host: AH4
H3K4:
  Actual: -0.1165
  Predicted: 0.7769
  Error: 0.8935
H3K4ac:
  Actual: 1.4110
  Predicted: 1.2990
  Error: 0.1120
H3K4me1:
  Actual: -2.2073
  Predicted: 0.0002
  Error: 2.2075
H3K4me2:
  Actual: -5.2983
  Predicted: -1.0600
  Error: 4.2383
H3K4me3:
  Actual: -3.5066
  Predicted: -1.2778
  Error: 2.2287
H3K9me3:
  Actual: -2.2073
  Predicted: -0.5124
  Error: 1.6949
H3R2me2a:
  Actual: -2.3330
  Predicted: -0.2125
  Error: 2.1205
H3R2me2s:
  Actual: -1.4271
